# Notebook 03 — Modelagem

## Tech Challenge Fase 1 — Classificação de SRAG (SIVEP-Gripe)

**Objetivo:** Treinar e otimizar múltiplos modelos de classificação para prever o desfecho clínico (Cura ou Óbito).

---

## Modelos implementados

| # | Modelo | Tipo | Por quê |
|---|--------|------|---------|
| 1 | Regressão Logística | Linear | Baseline simples e interpretável |
| 2 | Árvore de Decisão | Não-linear | Altamente interpretável, sem normalização |
| 3 | Random Forest | Ensemble | Robusto, captura não-linearidades |
| 4 | XGBoost | Gradient Boosting | Geralmente melhor performance |

## Estrutura do notebook
1. Configuração e carregamento dos dados processados
2. Treinamento individual de cada modelo
3. Avaliação no conjunto de validação
4. Comparativo e seleção do melhor modelo
5. Salvamento dos modelos

## 1. Configuração

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.metrics import f1_score, classification_report

warnings.filterwarnings('ignore')

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from src.tabular.modeling import definir_modelos, treinar_modelo, treinar_todos_modelos
from src.tabular.evaluation import calcular_metricas, comparar_modelos

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')
print('Configurado.')

Configurado.


## 2. Carregamento dos Dados Processados

> Execute o Notebook 02 antes para gerar os artefatos em `data/processed/`.

In [2]:
PROCESSED = ROOT / 'data' / 'processed'

X_train = pd.read_parquet(PROCESSED / 'X_train.parquet')
X_val   = pd.read_parquet(PROCESSED / 'X_val.parquet')
X_test  = pd.read_parquet(PROCESSED / 'X_test.parquet')
y_train = pd.read_parquet(PROCESSED / 'y_train.parquet').squeeze()
y_val   = pd.read_parquet(PROCESSED / 'y_val.parquet').squeeze()
y_test  = pd.read_parquet(PROCESSED / 'y_test.parquet').squeeze()

print(f'Treino:    X={X_train.shape}, y={y_train.shape} | Óbito: {y_train.mean()*100:.1f}%')
print(f'Validação: X={X_val.shape}, y={y_val.shape} | Óbito: {y_val.mean()*100:.1f}%')
print(f'Teste:     X={X_test.shape}, y={y_test.shape} | Óbito: {y_test.mean()*100:.1f}%')

Treino:    X=(168304, 37), y=(168304,) | Óbito: 8.6%
Validação: X=(36066, 37), y=(36066,) | Óbito: 8.6%
Teste:     X=(36066, 37), y=(36066,) | Óbito: 8.6%


## 3. Modelo 1 — Regressão Logística (Baseline)

**Justificativa:** Modelo linear simples, serve como baseline. Coeficientes são interpretáveis diretamente como log-odds. Requer normalização (já aplicada no preprocessing).

In [3]:
# Calcula scale_pos_weight para o XGBoost (equivalente ao class_weight='balanced')
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
scale_pos_weight = n_neg / n_pos
print(f'Desbalanceamento — Cura: {n_neg:,} | Óbito: {n_pos:,} | scale_pos_weight: {scale_pos_weight:.2f}')

modelos_config = definir_modelos(scale_pos_weight=scale_pos_weight)

# Treino com GridSearchCV (busca de hiperparâmetros)
# Para teste rápido, use usar_grid_search=False
lr = treinar_modelo('logistic_regression', modelos_config['logistic_regression'],
                    X_train, y_train, usar_grid_search=True)

# Avaliação na validação
metricas_lr = calcular_metricas('logistic_regression', lr, X_val, y_val)

Desbalanceamento — Cura: 153,794 | Óbito: 14,510 | scale_pos_weight: 10.60

[logistic_regression] Iniciando treinamento...
Fitting 5 folds for each of 8 candidates, totalling 40 fits


[logistic_regression] Melhores parâmetros: {'C': 1.0, 'solver': 'liblinear'}
[logistic_regression] F1 no CV (treino): 0.3729

[logistic_regression]
              precision    recall  f1-score   support

    Cura (0)       0.98      0.76      0.85     32957
   Óbito (1)       0.24      0.83      0.38      3109

    accuracy                           0.76     36066
   macro avg       0.61      0.79      0.61     36066
weighted avg       0.92      0.76      0.81     36066

ROC-AUC: 0.8640


In [4]:
# Visualizar os coeficientes mais importantes
coefs = pd.Series(lr.coef_[0], index=X_train.columns).sort_values(key=abs, ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
cores = ['#F44336' if v > 0 else '#2196F3' for v in coefs.values]
ax.barh(coefs.index[::-1], coefs.values[::-1], color=cores[::-1])
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('Coeficientes — Regressão Logística (Top 15)')
ax.set_xlabel('Coeficiente (log-odds)')
plt.tight_layout()
plt.savefig('../results/figures/coeficientes_logistic_regression.png', dpi=150)
plt.show()

## 4. Modelo 2 — Árvore de Decisão

**Justificativa:** Altamente interpretável — é possível visualizar o caminho de decisão. Não requer normalização. Tendência a overfitting sem regularização (controlamos via `max_depth`).

In [5]:
dt = treinar_modelo('decision_tree', modelos_config['decision_tree'],
                    X_train, y_train, usar_grid_search=True)

metricas_dt = calcular_metricas('decision_tree', dt, X_val, y_val)

print(f'\nProfundidade da árvore treinada: {dt.get_depth()}')
print(f'Número de folhas: {dt.get_n_leaves()}')


[decision_tree] Iniciando treinamento...
Fitting 5 folds for each of 24 candidates, totalling 120 fits


[decision_tree] Melhores parâmetros: {'criterion': 'entropy', 'max_depth': 10, 'min_samples_split': 50}
[decision_tree] F1 no CV (treino): 0.3953

[decision_tree]
              precision    recall  f1-score   support

    Cura (0)       0.98      0.78      0.87     32957
   Óbito (1)       0.27      0.83      0.40      3109

    accuracy                           0.79     36066
   macro avg       0.62      0.81      0.64     36066
weighted avg       0.92      0.79      0.83     36066

ROC-AUC: 0.8850

Profundidade da árvore treinada: 10
Número de folhas: 397


In [6]:
# Feature importance da Árvore
from src.tabular.evaluation import plotar_feature_importance
plotar_feature_importance('decision_tree', dt, list(X_train.columns))

Figura salva: /Users/arthuraugustopaulahardman/projetos/pos_fiap/fase1/tech-challenge-srag-v2/src/results/figures/feature_importance_decision_tree.png


## 5. Modelo 3 — Random Forest

**Justificativa:** Ensemble de múltiplas árvores com bagging. Reduz overfitting em relação à árvore única. Robusto a outliers e variáveis irrelevantes. Feature importance via média das impurezas.

In [7]:
rf = treinar_modelo('random_forest', modelos_config['random_forest'],
                    X_train, y_train, usar_grid_search=True)

metricas_rf = calcular_metricas('random_forest', rf, X_val, y_val)


[random_forest] Iniciando treinamento...
Fitting 5 folds for each of 12 candidates, totalling 60 fits


[random_forest] Melhores parâmetros: {'max_depth': None, 'min_samples_split': 10, 'n_estimators': 300}
[random_forest] F1 no CV (treino): 0.4978



[random_forest]
              precision    recall  f1-score   support

    Cura (0)       0.96      0.94      0.95     32957
   Óbito (1)       0.46      0.53      0.50      3109

    accuracy                           0.91     36066
   macro avg       0.71      0.74      0.72     36066
weighted avg       0.91      0.91      0.91     36066

ROC-AUC: 0.8940


In [8]:
plotar_feature_importance('random_forest', rf, list(X_train.columns))

Figura salva: /Users/arthuraugustopaulahardman/projetos/pos_fiap/fase1/tech-challenge-srag-v2/src/results/figures/feature_importance_random_forest.png


## 6. Modelo 4 — XGBoost

**Justificativa:** Gradient Boosting que constrói árvores sequencialmente, cada uma corrigindo os erros da anterior. Geralmente superior a Random Forest em dados tabulares. Possui regularização L1/L2 nativa.

In [9]:
xgb = treinar_modelo('xgboost', modelos_config['xgboost'],
                     X_train, y_train, usar_grid_search=True)

metricas_xgb = calcular_metricas('xgboost', xgb, X_val, y_val)


[xgboost] Iniciando treinamento...
Fitting 5 folds for each of 36 candidates, totalling 180 fits


[xgboost] Melhores parâmetros: {'learning_rate': 0.2, 'max_depth': 8, 'n_estimators': 300, 'subsample': 0.8}
[xgboost] F1 no CV (treino): 0.4482

[xgboost]
              precision    recall  f1-score   support

    Cura (0)       0.97      0.87      0.91     32957
   Óbito (1)       0.33      0.68      0.44      3109

    accuracy                           0.85     36066
   macro avg       0.65      0.78      0.68     36066
weighted avg       0.91      0.85      0.87     36066

ROC-AUC: 0.8761


In [10]:
plotar_feature_importance('xgboost', xgb, list(X_train.columns))

Figura salva: /Users/arthuraugustopaulahardman/projetos/pos_fiap/fase1/tech-challenge-srag-v2/src/results/figures/feature_importance_xgboost.png


## 7. Comparativo na Validação

In [11]:
resultados_val = [metricas_lr, metricas_dt, metricas_rf, metricas_xgb]
df_comparativo = comparar_modelos(resultados_val)

# Gráfico comparativo
metricas_plot = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
df_plot = df_comparativo[metricas_plot]

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(df_plot))
width = 0.15
cores = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']

for i, (metrica, cor) in enumerate(zip(metricas_plot, cores)):
    ax.bar(x + i * width, df_plot[metrica], width, label=metrica.upper(), color=cor, alpha=0.85)

ax.set_xlabel('Modelo')
ax.set_ylabel('Score')
ax.set_title('Comparativo de Métricas na Validação')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(df_plot.index, rotation=15)
ax.legend(loc='lower right')
ax.set_ylim(0, 1.1)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('../results/figures/comparativo_modelos_validacao.png', dpi=150)
plt.show()


Comparativo de modelos:
                     accuracy  precision  recall      f1  roc_auc
modelo                                                           
random_forest          0.9060     0.4612  0.5346  0.4952   0.8940
xgboost                0.8513     0.3269  0.6845  0.4425   0.8761
decision_tree          0.7883     0.2667  0.8321  0.4040   0.8850
logistic_regression    0.7622     0.2428  0.8298  0.3757   0.8640


## 8. Salvamento dos Modelos

In [12]:
MODELS_DIR = ROOT / 'results' / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

modelos_treinados = {
    'logistic_regression': lr,
    'decision_tree': dt,
    'random_forest': rf,
    'xgboost': xgb,
}

for nome, modelo in modelos_treinados.items():
    caminho = MODELS_DIR / f'{nome}.pkl'
    joblib.dump(modelo, caminho)
    print(f'Salvo: {caminho}')

print('\nTodos os modelos salvos em results/models/')

Salvo: /Users/arthuraugustopaulahardman/projetos/pos_fiap/fase1/tech-challenge-srag-v2/results/models/logistic_regression.pkl
Salvo: /Users/arthuraugustopaulahardman/projetos/pos_fiap/fase1/tech-challenge-srag-v2/results/models/decision_tree.pkl


Salvo: /Users/arthuraugustopaulahardman/projetos/pos_fiap/fase1/tech-challenge-srag-v2/results/models/random_forest.pkl
Salvo: /Users/arthuraugustopaulahardman/projetos/pos_fiap/fase1/tech-challenge-srag-v2/results/models/xgboost.pkl

Todos os modelos salvos em results/models/


## 9. Conclusões da Modelagem

**Análise dos resultados na validação:**

> *Preencha após executar as células acima com os valores reais obtidos.*

**Escolha da métrica principal — F1-score:**

Em um contexto médico de classificação de óbito, priorizamos o **F1-score** como métrica principal porque:
- O dataset é **desbalanceado** (mais curas que óbitos)
- **Falsos negativos** (predizer cura quando é óbito) têm custo clínico alto
- O F1 equilibra Precision e Recall, sendo mais informativo que Accuracy
- Complementado pelo **ROC-AUC** para avaliar o poder discriminativo geral

**Próximos passos:** Notebook 04 — Avaliação e Interpretabilidade (conjunto de TESTE)